# FAST-UAV - Hybrid VTOL Design Optimization
*Author: Félix Pollet - 2023* <br>

This notebook provides an example of design optimization of a hybrid VTOL UAV. For now, only the quadplane configuration is covered. It consists of a fixed-wing UAV with four VTOL rotors mounted on each side of the wings. The workflow is the same as for multirotor and fixed-wing UAVs, except that there are more variables to deal with.

## 🛩️ Skywalker X8 스타일 하이브리드 VTOL 설계 가이드

### 시스템 구성 요소

**Skywalker X8 유형** (Quadplane Configuration):
1. **고정익 (Fixed-Wing)**
   - 날개 (Wing): 양력 발생, 순항 비행
   - 수평 꼬리날개 (Horizontal Tail): 피치 안정성
   - 수직 꼬리날개 (Vertical Tail): 요 안정성
   - Pusher 프로펠러: 전진 추력 (뒤쪽 배치)

2. **멀티로터 (VTOL 시스템)**
   - 4개 프로펠러: 수직 이착륙 및 호버링
   - 날개 양쪽에 장착된 암(arms)에 부착
   - 이륙/착륙 시에만 사용

3. **추진 시스템 2개**
   - **Fixed-Wing Propulsion**: Pusher 모터, 프로펠러, ESC, 배터리
   - **Multirotor Propulsion**: VTOL 모터(×4), 프로펠러(×4), ESC(×4), 배터리

### 비행 모드 전환 시나리오

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐
│ VTOL 모드   │ --> │  전환 모드   │ --> │ 고정익 모드  │
│ (이륙/호버) │     │ (가속/감속)  │     │   (순항)     │
└─────────────┘     └──────────────┘     └──────────────┘
   4 로터 ON         4 로터 + Pusher       Pusher ON
   Pusher OFF        모두 ON               4 로터 OFF
```

### FAST-UAV 모델링 구조

```yaml
# hybrid_mdo.yaml 설정

model:
  scenarios:
    id: fastuav.scenarios.hybrid  # 하이브리드 시나리오
    
  propulsion:
    id: fastuav.propulsion.hybrid  # 2개의 추진 시스템
    gearbox: False                 # 기어박스 미사용
    
  geometry:
    id: fastuav.geometry.hybrid    # 날개 + 멀티로터 암
    
  structures:
    id: fastuav.structures.hybrid  # 날개 스파 + 암 구조
    spar_model: "I_beam"           # I-beam 또는 pipe
    
  aerodynamics:
    id: fastuav.aerodynamics.hybrid  # 양력, 항력 계산
    
  stability:
    id: fastuav.stability.hybrid   # 정적 안정성 (Static Margin)
    
  performance:
    missions:
      id: fastuav.performance.mission
      file_path: ../missions/missions_hybrid.yaml
```

### 미션 프로파일 정의

```yaml
# missions_hybrid.yaml

routes:
  main_route:
    climb_part:
      phase_id: multirotor_climb  # VTOL 로터로 상승
    hover_part:
      phase_id: hover             # 호버링 (전환 준비)
    cruise_part:
      phase_id: fixedwing_cruise  # Pusher로 순항
```

### 주요 설계 변수

**1. 고정익 추진 (Pusher)**
- `propeller:beta` (0.48~1.0): Pusher 프로펠러 피치/직경 비
- `propeller:advance_ratio:cruise`: 순항 전진비
- `motor:torque:k`: 모터 토크 계수
- `battery:energy:k`: 배터리 에너지 계수

**2. 멀티로터 추진 (VTOL)**
- `propeller:beta` (0.3~0.6): VTOL 프로펠러 피치/직경 비
- `propeller:advance_ratio:climb`: 상승 전진비
- `motor:torque:k`: VTOL 모터 토크 계수

**3. 기하학**
- `wing:AR` (8~20): 날개 가로세로비
- `wing:lambda` (0.1~1.0): 날개 테이퍼비
- `tail:horizontal:AR` (3~6): 수평꼬리 가로세로비
- `tail:vertical:AR` (0.9~4): 수직꼬리 가로세로비

**4. 구조**
- `wing:spar:depth:k`: 날개 스파 깊이 계수
- `arms:diameter:k`: VTOL 암 직경 비율

**5. 공기역학**
- `aerodynamics:CD0:guess` (0.01~1.0): 기생 항력 계수

### 제약 조건

**추진 시스템 제약**
- 프로펠러 회전수 한계
- 모터 토크 한계
- 배터리 전력/전압 한계
- ESC 전력 한계

**구조 제약**
- 날개 스파 응력 (VTOL 이륙 하중)
- 암 구조 응력

**안정성 제약**
- Static Margin (최소/최대값)

**기하 제약**
- VTOL 프로펠러 위치 (날개 끝 이내)
- 동체 부피 요구사항

In [ ]:
# ========================================
# Skywalker X8 스타일 설계 예시 파라미터
# ========================================

skywalker_x8_style = {
    "🛩️ 시스템 개요": {
        "유형": "Quadplane (고정익 + 쿼드콥터)",
        "추진_시스템": "2개 (Pusher + VTOL ×4)",
        "날개": "Flying Wing 또는 일반 날개",
        "전환_모드": "수직 이착륙 ↔ 수평 순항",
    },
    
    "⚙️ 고정익 추진 (Pusher)": {
        "위치": "후방 (Pusher configuration)",
        "용도": "순항 비행",
        "프로펠러_직경": "~0.25-0.35 m",
        "모터_Kv": "낮음 (고효율, 500-1000)",
        "프로펠러_피치": "높음 (0.6-1.0)",
    },
    
    "🚁 멀티로터 추진 (VTOL)": {
        "개수": "4개 (쿼드콥터)",
        "위치": "날개 양쪽 (arms에 장착)",
        "용도": "이륙, 착륙, 호버",
        "프로펠러_직경": "~0.20-0.30 m",
        "모터_Kv": "중간 (1000-2000)",
        "프로펠러_피치": "중간 (0.3-0.5)",
    },
    
    "📏 기하학 파라미터": {
        "날개_스팬": "1.0-2.0 m",
        "날개_면적": "0.3-0.6 m²",
        "날개_가로세로비_AR": "8-15",
        "총_중량_MTOW": "2-5 kg",
    },
    
    "🔋 배터리 시스템": {
        "전압": "22.2V (6S) 또는 44.4V (12S)",
        "용량": "5000-10000 mAh",
        "타입": "LiPo",
        "배터리_개수": "1개 (공용) 또는 2개 (분리)",
    },
    
    "🎯 미션 프로파일": {
        "1단계_이륙": "VTOL 모터만 사용 (4개)",
        "2단계_상승": "VTOL 모터로 100-150m 상승",
        "3단계_전환": "VTOL + Pusher 동시 작동",
        "4단계_순항": "Pusher만 사용 (고정익 비행)",
        "5단계_복귀": "Pusher로 귀환",
        "6단계_착륙": "감속 → VTOL 전환 → 착륙",
    }
}

print("=" * 70)
print("🛩️ SKYWALKER X8 스타일 하이브리드 VTOL 설계 가이드")
print("=" * 70)

for category, params in skywalker_x8_style.items():
    print(f"\n{category}")
    print("-" * 70)
    for key, value in params.items():
        print(f"  {key:20s}: {value}")

print("\n" + "=" * 70)
print("🔧 FAST-UAV에서 모델링하는 방법")
print("=" * 70)

modeling_steps = """
1️⃣ 구성 파일 선택:
   - hybrid_mdo.yaml 사용
   - 2개의 추진 시스템 자동 설정

2️⃣ 미션 정의 (missions_hybrid.yaml):
   routes:
     main_route:
       climb_part:
         phase_id: multirotor_climb  # VTOL로 상승
       hover_part:
         phase_id: hover             # 호버 (전환)
       cruise_part:
         phase_id: fixedwing_cruise  # Pusher로 순항

3️⃣ 입력 파일 설정 (problem_inputs.xml):
   - mission:sizing:payload:mass = 1.0 kg (페이로드)
   - mission:sizing:main_route:cruise:distance = 10000 m
   - mission:sizing:main_route:cruise:altitude = 150 m
   - data:propulsion:multirotor:propeller:number = 4
   - data:geometry:wing:AR = 10-15 (고효율)

4️⃣ 최적화 목표 설정:
   # 질량 최소화
   objective:
     - name: data:weight:mtow
   
   # 또는 항속시간 최대화
   objective:
     - name: data:performance:endurance:cruise
       scaler: -1e-3

5️⃣ 실행:
   optim_problem = oad.optimize_problem(CONFIGURATION_FILE)

6️⃣ 결과 확인:
   - 최적 MTOW
   - Pusher 프로펠러 직경, 모터 스펙
   - VTOL 프로펠러 직경, 모터 스펙
   - 날개 형상 (스팬, 면적, AR)
   - 배터리 용량/전압
"""

print(modeling_steps)

print("=" * 70)
print("⚠️ 주의사항")
print("=" * 70)
print("""
1. VTOL 프로펠러 위치가 날개 끝을 넘지 않도록 제약 필요
2. Static Margin 확인 (안정성)
3. 날개 스파가 VTOL 이륙 하중을 견딜 수 있는지 확인
4. 전환 모드는 현재 자동 모델링되지 않음 (수동 분석 필요)
5. 배터리가 2개 시스템을 모두 지원할 수 있는지 확인
""")

## 1. Setting up and analyzing the initial problem

In [ ]:
# Import required librairies
import os.path as pth
import openmdao.api as om
import logging
import warnings
import shutil
import fastoad.api as oad
from time import time
import matplotlib.pyplot as plt
from fastuav.utils.postprocessing.analysis_and_plots import *

# Declare paths to folders and files
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "hybrid_mdo.yaml")
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_hybrid.xml")  # Source file provided for the example

# For having log messages display on screen
#logging.basicConfig(level=logging.INFO, format="%(levelname)-8s: %(message)s")
#warnings.filterwarnings(action="ignore")

# For using all screen width
from IPython.display import display, HTML, IFrame
display(HTML("<style>.container { width:95% !important; }</style>"))

하이브리드 VTOL 무인 항공기는 고정익 무인 항공기와 유사한 전진 추진 시스템과 VTOL 추진(이륙 및 호버링)을 위한 두 가지 추진 시스템을 갖춘 고정익 구조로 구성되어 있습니다. 4개의 로터로 구성된 VTOL 추진은 날개에 부착된 두 개의 암으로 지원됩니다. 다음 N2 다이어그램에서 새로운 구성 요소를 시각화할 수 있습니다.

In [ ]:
N2_FILE = pth.join(WORK_FOLDER_PATH, "n2.html")
oad.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)
from IPython.display import IFrame
IFrame(src=N2_FILE, width="100%", height="500px")

Let's generate the input file from the reference source file provided for the example:

In [ ]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

Complete or modify the input values if necessary:

In [ ]:
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

## 2. Running an MDO

You can have a look at the optimization problem definition in the [configuration file]. You may want to change the objective function for example, or keep the one that is defined. Then, run the optimization process:

In [ ]:
optim_problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)

Let's save and visualize the results:

In [ ]:
OUTPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_outputs.xml")
HYBRID_OUTPUT_FILE = pth.join(SOURCE_FOLDER_PATH, 'problem_outputs_HybridVTOL_mdo.xml')
shutil.copy(OUTPUT_FILE, HYBRID_OUTPUT_FILE)

In [ ]:
oad.optimization_viewer(CONFIGURATION_FILE)

In [ ]:
oad.variable_viewer(OUTPUT_FILE)

## 3. Analysis and plots

You can now use postprocessing plots to visualize the results of the MDO.

In [1]:
fig = fixedwing_geometry_plot(OUTPUT_FILE)  # In progress: the geometry plot for hybrid VTOL still has to be developed. Only the fixed-wing part is disiplayed
fig.show()

NameError: name 'fixedwing_geometry_plot' is not defined

In [ ]:
fig = mass_breakdown_sun_plot_drone(OUTPUT_FILE)
fig.show()